# Apply a trained COMIND model to new data

Reconstruct a fitted SubtypingEM from an experiment `.npz` (no re-fit) and
score a longitudinal table with the **same frozen** preprocessing used at
train time.

This notebook targets the July 2026 K82 / lnB_lnC runs. Those NPZs store
lognorm `mu`/`sigma` but **not** `per_biomarker_min_shift`, so the affine
shift is recovered once from a training-reference CSV using the same
winsor → sign-flip → min procedure as `run_comind_ppmi.py`.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from COMIND_transformer.subtyping_em_transformer import SubtypingEM
from COMIND_transformer.utils import solve_system
from COMIND_transformer.preprocessing import (
    build_connectome,
    lognormal_to_uniform,
    parse_data,
)

# --- paths ---
NPZ_PATH = (
    "/home/dsemchin/COMIND/experiments/2026_07_02_PPMI_K82_cold_gs_lowscalar_betaclin_betaall/"
    "results/PPMI_subtyping_grid_betajsd_274_lambda_f10p000_lambda_kappa5p000_lambda_cog0p000_"
    "lambda_scalar10p000_scalar_K_center0p010_lambda_jsd10p000_lambda_beta0p000_n_subtypes4.npz"
)
CONNECTOME_PATH = "/home/dsemchin/data/iit_connectivity_matrix/K_82_top10_normalized.csv"
NEW_DATA_CSV = "/home/dsemchin/data/PPMI_deviation_scores.csv"
# Used only to recover the affine shift (not stored in this npz generation).
# Prefer the original training table; same file is fine for this PPMI demo.
TRAIN_REF_CSV = NEW_DATA_CSV

SUBJ_ID_COL = "subj_id"
TIME_COL = "time"
# PPMI deviation table stores months since baseline; training divided by 12.
TIME_IN_MONTHS = True

CLINICAL_COLS = ["MCATOT", "TD_score", "PIGD_score"]
COG_COLS = [f"{c}__raw" for c in CLINICAL_COLS]
Z_SIGN = -1
WINSOR_LIMIT = 6.0
MOCA_CEILING, TD_SCORE_CEILING, PIGD_SCORE_CEILING = 30.0, 20.0, 16.0

### 1. Load the trained model -- reconstruct SubtypingEM, do NOT re-fit

In [2]:
result = np.load(NPZ_PATH, allow_pickle=True)
print("npz keys:", sorted(result.files))

cluster_f = np.asarray(result["cluster_f"])  # (n_subtypes, n_biomarkers)
cluster_cog_a = np.asarray(result["cluster_cog_a"])
cluster_cog_b = np.asarray(result["cluster_cog_b"])
final_s = np.asarray(result["final_s"])
final_scalar_K = float(result["final_scalar_K"])
final_kappa = np.asarray(result["final_kappa"])
biomarker_names = list(result["biomarker_names"])
n_subtypes = int(result["n_subtypes"])
n_biomarkers = cluster_f.shape[1]
imaging_cols = [b for b in biomarker_names if b not in CLINICAL_COLS]
assert len(imaging_cols) + len(CLINICAL_COLS) == n_biomarkers

apply_lognorm_connectome = bool(result["apply_lognorm_connectome"])
apply_lognorm_clinical = bool(result["apply_lognorm_clinical"])

# Frozen lognorm params (saved). Affine shift was NOT saved in this npz gen.
lognorm_connectome_mu = lognorm_connectome_sigma = None
lognorm_clinical_mu = lognorm_clinical_sigma = None
if apply_lognorm_connectome and len(result["lognorm_connectome_mu"]):
    lognorm_connectome_mu = np.asarray(result["lognorm_connectome_mu"], dtype=float)
    lognorm_connectome_sigma = np.asarray(result["lognorm_connectome_sigma"], dtype=float)
if apply_lognorm_clinical and len(result["lognorm_clinical_mu"]):
    lognorm_clinical_mu = np.asarray(result["lognorm_clinical_mu"], dtype=float)
    lognorm_clinical_sigma = np.asarray(result["lognorm_clinical_sigma"], dtype=float)

# Recover per-biomarker affine shift from the training-reference table.
# Same recipe as run_comind_ppmi.apply_biomarker_transforms on the complete-case cohort.
def recover_affine_shift(ref_csv, imaging_cols, clinical_cols):
    df = pd.read_csv(ref_csv).replace([np.inf, -np.inf], np.nan)
    required = list(imaging_cols) + list(clinical_cols)
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"TRAIN_REF_CSV missing columns: {missing[:5]}")

    # Match training driver when PDSTATE is present.
    if "PDSTATE" in df.columns:
        group_has_off = df.groupby([SUBJ_ID_COL, TIME_COL])["PDSTATE"].transform(
            lambda s: "OFF" in s.values
        )
        drop_on = (df["PDSTATE"] == "ON") & group_has_off
        df = df.loc[~drop_on].copy()
        dup_mask = df.duplicated(subset=[SUBJ_ID_COL, TIME_COL], keep=False)
        if dup_mask.any():
            def _coalesce(group):
                out = {}
                for col in group.columns:
                    non_null = group[col].dropna()
                    out[col] = non_null.iloc[0] if len(non_null) else np.nan
                return pd.Series(out)

            df = (
                df.sort_values([SUBJ_ID_COL, TIME_COL])
                .groupby([SUBJ_ID_COL, TIME_COL], as_index=False)
                .apply(_coalesce, include_groups=False)
                .reset_index(drop=True)
            )

    df = df.loc[df[required].notna().all(axis=1)].copy()
    z_raw = np.clip(df[imaging_cols].to_numpy(dtype=float), -WINSOR_LIMIT, WINSOR_LIMIT)
    z_signed = Z_SIGN * z_raw
    shift = z_signed.min(axis=0)
    print(
        f"Recovered affine shift from {len(df)} complete-case rows "
        f"({df[SUBJ_ID_COL].nunique()} subjects); "
        f"range [{shift.min():.3f}, {shift.max():.3f}]"
    )
    return shift


if "per_biomarker_min_shift" in result.files and len(result["per_biomarker_min_shift"]):
    per_biomarker_min_shift = np.asarray(result["per_biomarker_min_shift"], dtype=float)
    print("Loaded per_biomarker_min_shift from npz")
else:
    print("per_biomarker_min_shift not in npz -- recovering from TRAIN_REF_CSV")
    per_biomarker_min_shift = recover_affine_shift(
        TRAIN_REF_CSV, imaging_cols, CLINICAL_COLS
    )

t_max = 40
t_step = 0.01
t_span = np.linspace(0, t_max, int(t_max / t_step))

em = SubtypingEM(
    K=None,
    t_max=t_max,
    step=t_step,
    n_subtypes=n_subtypes,
    lambda_cog=0.0,
    verbose=0,
)
em.cluster_f = [np.ravel(cluster_f[k]) for k in range(n_subtypes)]
em.cluster_cog_a = [np.ravel(cluster_cog_a[k]) for k in range(n_subtypes)]
em.cluster_cog_b = [float(cluster_cog_b[k]) for k in range(n_subtypes)]
em.final_s = final_s
em.final_scalar_K = final_scalar_K
em.final_kappa = final_kappa
em.t_span = t_span

print(f"Loaded {n_subtypes}-subtype model, {n_biomarkers} biomarkers "
      f"({len(imaging_cols)} imaging + {len(CLINICAL_COLS)} clinical)")
print(
    f"Lognorm PIT at train time -- connectome: {apply_lognorm_connectome}, "
    f"clinical: {apply_lognorm_clinical}"
)

npz keys: ['theta_history', 'cog_history', 'beta_history', 'kappa_history', 'final_kappa', 'lse_history', 'iter_times', 'accepted_solver_stages', 'assign_changes', 'assignment_history', 'beta_val', 'beta_val_with_cog', 'beta_val_no_cog', 'val_assignments_with_cog', 'val_assignments_no_cog', 'val_lse_with_cog', 'val_lse_no_cog', 'candidate', 'experiment_name', 'warm_start_npz', 'warm_start_format', 'jitter_applied', 'jitter_strength', 'jitter_seed', 'skip_cv', 'cv_parallel', 'cv_workers', 'checkpoint_path', 'max_iter', 'theta_solver_stages', 'f_init', 'train_assignments', 'val_assignments', 'train_ids', 'val_ids', 'final_assignments', 'cluster_f', 'cluster_cog_a', 'cluster_cog_b', 'final_scalar_K', 'final_s', 'biomarker_names', 'n_subtypes', 'n_subtypes_list', 'lambda_f', 'lambda_cog', 'lambda_scalar', 'scalar_K_center', 'lambda_kappa', 'lambda_jsd', 'lambda_beta', 'param_grid_size', 'n_hyper_per_K', 'k_idx', 'sub_cand', 'bic', 'bic_neg2_log_L', 'bic_penalty', 'n_obs', 'bic_n_params', '

KeyError: 'per_biomarker_min_shift is not a file in the archive'

### 2. Load K, aligned to the model's biomarker order

In [ ]:
# Align K to biomarker_names order; pad clinical (and any other missing) as
# disconnected nodes -- same geometry as pad_connectome_matrix at train time.
K, disconnected = build_connectome(
    CONNECTOME_PATH, biomarker_names, pad_missing=True
)
assert K.shape == (n_biomarkers, n_biomarkers)
em.K = K
print(f"K aligned: {K.shape}, disconnected: {disconnected}")

### 3. Transform NEW raw data using the FROZEN training-time preprocessing

Do **not** re-fit the affine shift or lognorm on the new table. Winsor limit,
sign flip, recovered `per_biomarker_min_shift`, and stored lognorm `mu`/`sigma`
are fixed. Clinical scores are also part of `X_obs` (disconnected nodes) for
this model, with raw copies kept for `cog` (needed by `transform` even when
`use_cognitive_prior=False`).

In [ ]:
def transform_new_data(df, per_biomarker_min_shift):
    """Apply frozen training-time transforms; write X features into df columns."""
    df = df.copy()
    missing = [c for c in biomarker_names if c not in df.columns]
    if missing:
        raise ValueError(f"New data missing model biomarkers: {missing[:5]}")

    # Keep raw clinical for cog before overwriting CLINICAL_COLS in X_obs.
    for raw_name, col in zip(COG_COLS, CLINICAL_COLS):
        df[raw_name] = df[col].to_numpy(dtype=float)

    z_raw = np.clip(df[imaging_cols].to_numpy(dtype=float), -WINSOR_LIMIT, WINSOR_LIMIT)
    z_signed = Z_SIGN * z_raw
    z_transformed = np.maximum(0.0, z_signed - per_biomarker_min_shift)
    if lognorm_connectome_mu is not None:
        z_transformed = lognormal_to_uniform(
            z_transformed, lognorm_connectome_mu, lognorm_connectome_sigma
        )
    df[imaging_cols] = z_transformed

    moca_t = (MOCA_CEILING - df["MCATOT"].to_numpy(dtype=float)) / (
        MOCA_CEILING / WINSOR_LIMIT
    )
    td_t = df["TD_score"].to_numpy(dtype=float) / (TD_SCORE_CEILING / WINSOR_LIMIT)
    pigd_t = df["PIGD_score"].to_numpy(dtype=float) / (PIGD_SCORE_CEILING / WINSOR_LIMIT)
    clinical_stacked = np.column_stack([moca_t, td_t, pigd_t])
    if lognorm_clinical_mu is not None:
        clinical_stacked = lognormal_to_uniform(
            clinical_stacked, lognorm_clinical_mu, lognorm_clinical_sigma
        )
    df[CLINICAL_COLS] = clinical_stacked

    if TIME_IN_MONTHS:
        df[TIME_COL] = df[TIME_COL].to_numpy(dtype=float) / 12.0

    return df


df_new = pd.read_csv(NEW_DATA_CSV).replace([np.inf, -np.inf], np.nan)

# Light cleaning so parse_data does not raise on NaNs in X / clinical.
# (Full PDSTATE / duplicate merge is optional for brand-new cohorts.)
if "PDSTATE" in df_new.columns:
    group_has_off = df_new.groupby([SUBJ_ID_COL, TIME_COL])["PDSTATE"].transform(
        lambda s: "OFF" in s.values
    )
    df_new = df_new.loc[~((df_new["PDSTATE"] == "ON") & group_has_off)].copy()

df_new = df_new.dropna(subset=biomarker_names).copy()
df_new = transform_new_data(df_new, per_biomarker_min_shift)

print(
    f"Prepared {len(df_new)} rows / {df_new[SUBJ_ID_COL].nunique()} subjects; "
    f"X_obs cols = {len(biomarker_names)}"
)
print(
    f"X imaging range: [{df_new[imaging_cols].to_numpy().min():.4f}, "
    f"{df_new[imaging_cols].to_numpy().max():.4f}]"
)

### 4. Build patient list, run transform (no fitting)

In [ ]:
# clinical_cols must be length-3 for this model: transform() always does
# cog @ cluster_cog_a even when use_cognitive_prior=False (extra term is
# zeroed via lambda, but the matmul still runs).
patients = parse_data(
    df_new,
    subj_id_col=SUBJ_ID_COL,
    time_col=TIME_COL,
    X_cols=biomarker_names,
    clinical_cols=COG_COLS,
    metadata_cols=[],
)

tr = em.transform(patients, use_cognitive_prior=False)
beta_new = np.asarray(tr["beta"], dtype=float)
subtype_new = np.asarray(tr["subtype"], dtype=int)

print(f"Transformed {len(patients)} patients")
print(
    f"beta: min={beta_new.min():.2f}, max={beta_new.max():.2f}, "
    f"mean={beta_new.mean():.2f}"
)
unique, counts = np.unique(subtype_new, return_counts=True)
print("Subtype counts:", dict(zip(unique.tolist(), counts.tolist())))

### 5. Plots

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(beta_new, bins=20, edgecolor="k")
plt.xlabel("beta (estimated disease time)")
plt.ylabel("count")
plt.title("New-data beta distribution")
plt.tight_layout()
plt.show()

In [ ]:
# Spaghetti: patient trajectories overlaid on fitted subtype curves
subtype_colors = plt.cm.tab10(np.arange(n_subtypes) / 10.0)
pid_to_beta = {p["id"]: float(beta_new[i]) for i, p in enumerate(patients)}
pid_to_subtype = {p["id"]: int(subtype_new[i]) for i, p in enumerate(patients)}

x0 = np.zeros(n_biomarkers)
traj_by_subtype = [
    solve_system(x0, cluster_f[k], K, t_span, final_scalar_K, final_kappa)
    for k in range(n_subtypes)
]

# Prefer a few clinically interesting ROIs if present; else first imaging cols.
preferred = [
    "L_entorhinal_Z_predict",
    "L_middletemporal_Z_predict",
    "L_inferiortemporal_Z_predict",
    "MCATOT",
    "TD_score",
    "PIGD_score",
]
roi_idx = [biomarker_names.index(n) for n in preferred if n in biomarker_names]
if not roi_idx:
    roi_idx = list(range(min(6, n_biomarkers)))

fig, axes = plt.subplots(1, len(roi_idx), figsize=(4 * len(roi_idx), 4), sharex=True)
if len(roi_idx) == 1:
    axes = [axes]

for ax, ridx in zip(axes, roi_idx):
    for p in patients:
        pid = p["id"]
        sub = pid_to_subtype[pid]
        t_vals = p["dt"] + pid_to_beta[pid]
        y_vals = p["X_obs"][:, ridx]
        if len(t_vals) < 2:
            continue
        order = np.argsort(t_vals)
        ax.plot(
            t_vals[order],
            y_vals[order],
            color=subtype_colors[sub],
            alpha=0.35,
            lw=1.2,
        )
    for k in range(n_subtypes):
        ax.plot(
            t_span,
            final_s[ridx] * traj_by_subtype[k][ridx],
            color=subtype_colors[k],
            lw=2,
        )
    ax.set_title(biomarker_names[ridx], fontsize=9)
    ax.set_xlabel("disease time")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()